In [14]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
plt.rc('figure', figsize=(20, 5))

try:
    from itertools import izip_longest
except ImportError:  # Python 3
    from itertools import zip_longest as izip_longest

    
def grouper(iterable, n, fillvalue=None):
    args = [iter(iterable)] * n
    return izip_longest(*args, fillvalue=fillvalue)
    
def base_to_index(base):
    if base == 'A':
        return(1)
    elif base == 'C':
        return(2)
    elif base == 'G':
        return(3)
    elif base == 'T':
        return(4)
    elif base == 'N':
        return(5)
    elif base == '-':
        return(6)
    raise NameError("Unknown base : " + base)


def site_to_edit_calls(captured_sequence,known_sequence):
    assert(len(captured_sequence) == len(known_sequence))
    editing_pattern = np.zeros(7*26).reshape((7, 26))
    for (index,(cap,ref)) in enumerate(zip(captured_sequence,known_sequence)):
        if cap == ref:
            editing_pattern[0,index] += 1
        else:
            editing_pattern[base_to_index(cap),index] += 1
    return(editing_pattern)
      
wt_t7 =   'CCAGGAAGTACTCGAGTACTTCCTGG'
wt_rnf2 = 'CCTGTCATCTTAGCTAAGATGACAGG'

samples = ["p7_PigUMI_8_CTCTTCTGCT","p7_PigUMI_ct_GGCGCCTTAA","p7_PigUMI_pRDA_CGAGGTATAA","p7_PigUMI_pTpR8_GCTGCTGGTA"]
for sample in samples:
    print("Processing sample: " + sample)
    
    base_edits = {}
    base_edits[wt_t7] = np.zeros(7*26).reshape((7, 26))
    base_edits[wt_rnf2] = np.zeros(7*26).reshape((7, 26))

    total_reads = 0
    with open(sample + '_sequences.fasta', 'r') as infile:
        for line_set in grouper(infile, 4, ''):
            total_reads += 1
            targets = line_set[1]
            # print(targets)
            t7_captured = targets[0:26]
            rnf2_captured = targets[26:52]
            # print(len(t7_captured))
            base_edits[wt_t7] = base_edits[wt_t7] + site_to_edit_calls(t7_captured,wt_t7)
            base_edits[wt_rnf2] = base_edits[wt_rnf2] + site_to_edit_calls(rnf2_captured,wt_rnf2)

    normalized = base_edits[wt_rnf2]/base_edits[wt_rnf2].T.sum(axis=1)
    df = pd.DataFrame(normalized, index=['WT','A','C','G','T','N','-'],columns=[x for x in wt_rnf2])
    df = df.drop(['WT'])

    myploto = sns.heatmap(df,cmap="YlOrBr",annot=True, fmt=".1%")
    myploto.set(title=sample + "_RNF2_read_count_" + str(total_reads))
    fig = myploto.get_figure()
    fig.savefig(sample + '_RNF2.png')
    fig.clear()
    
    normalized = base_edits[wt_t7]/base_edits[wt_t7].T.sum(axis=1)
    df = pd.DataFrame(normalized, index=['WT','A','C','G','T','N','-'],columns=[x for x in wt_t7])
    df = df.drop(['WT'])
    myploto = sns.heatmap(df,cmap="YlOrBr",annot=True, fmt=".1%")
    myploto.set(title=sample + "_T7_" + str(total_reads))
    fig = myploto.get_figure()
    fig.savefig(sample + '_T7.png')
    fig.clear()
    

Processing sample: p7_PigUMI_8_CTCTTCTGCT
Processing sample: p7_PigUMI_ct_GGCGCCTTAA
Processing sample: p7_PigUMI_pRDA_CGAGGTATAA
Processing sample: p7_PigUMI_pTpR8_GCTGCTGGTA


<Figure size 1440x360 with 0 Axes>